<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">


# Procesamiento de lenguaje natural
## Custom embedddings con Gensim



### Objetivo
El objetivo es utilizar documentos / corpus para crear embeddings de palabras basado en ese contexto. Se utilizará canciones de bandas para generar los embeddings, es decir, que los vectores tendrán la forma en función de como esa banda haya utilizado las palabras en sus canciones.

In [1]:
import pandas as pd

# try:
#  from gensim.models import Word2Vec
# except:
#  !pip install gensim
#  from gensim.models import Word2Vec

from gensim.models import Word2Vec

### Datos
Utilizaremos como dataset canciones de bandas de habla inglesa.

In [2]:
# Ejecutar solo si estas en Colab!

# Descargar la carpeta de dataset
# import os
# import platform
# if os.access('./songs_dataset', os.F_OK) is False:
#    if os.access('songs_dataset.zip', os.F_OK) is False:
#        if platform.system() == 'Windows':
#            !curl https://raw.githubusercontent.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/main/datasets/songs_dataset.zip -o songs_dataset.zip
#        else:
#            !wget songs_dataset.zip https://github.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/raw/main/datasets/songs_dataset.zip
#    !unzip -q songs_dataset.zip
# else:
#    print("El dataset ya se encuentra descargado")

In [3]:
import zipfile
import os

output_zip = "../datasets/songs_dataset.zip"
extract_dir = "./"
with zipfile.ZipFile(output_zip, "r") as zip_ref:
    zip_ref.extractall(extract_dir)

In [4]:
# Posibles bandas
os.listdir("./songs_dataset/")

['amy-winehouse.txt',
 'leonard-cohen.txt',
 'r-kelly.txt',
 'blink-182.txt',
 'nirvana.txt',
 'britney-spears.txt',
 'nursery_rhymes.txt',
 'kanye-west.txt',
 'patti-smith.txt',
 'nicki-minaj.txt',
 'prince.txt',
 'lil-wayne.txt',
 'bob-dylan.txt',
 'disney.txt',
 'radiohead.txt',
 'al-green.txt',
 'bieber.txt',
 'bruce-springsteen.txt',
 'johnny-cash.txt',
 'jimi-hendrix.txt',
 'dj-khaled.txt',
 'lorde.txt',
 'janisjoplin.txt',
 'kanye.txt',
 'bob-marley.txt',
 'paul-simon.txt',
 'alicia-keys.txt',
 'dickinson.txt',
 'bruno-mars.txt',
 'lady-gaga.txt',
 'eminem.txt',
 'adele.txt',
 'notorious_big.txt',
 'bjork.txt',
 'michael-jackson.txt',
 'beatles.txt',
 'Kanye_West.txt',
 'nickelback.txt',
 'dr-seuss.txt',
 'missy-elliott.txt',
 'notorious-big.txt',
 'drake.txt',
 'dolly-parton.txt',
 'joni-mitchell.txt',
 'ludacris.txt',
 'Lil_Wayne.txt',
 'rihanna.txt',
 'lin-manuel-miranda.txt',
 'cake.txt']

In [ ]:
# Armar el dataset utilizando salto de línea para separar las oraciones/docs
df = pd.read_csv("songs_dataset/beatles.txt", sep="/n", header=None)
df.head()

/tmp/ipykernel_413756/148169198.py:2: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df = pd.read_csv('songs_dataset/beatles.txt', sep='\\n', header=None)


,0
0,"Yesterday, all my troubles seemed so far away"
1,Now it looks as though they're here to stay
2,"Oh, I believe in yesterday Suddenly, I'm not h..."
3,There's a shadow hanging over me.
4,"Oh, yesterday came suddenly Why she had to go ..."


In [6]:
print("Cantidad de documentos:", df.shape[0])

Cantidad de documentos: 1846


### 1 - Preprocesamiento

In [7]:
from tensorflow.keras.preprocessing.text import text_to_word_sequence

sentence_tokens = []
# Recorrer todas las filas y transformar las oraciones
# en una secuencia de palabras (esto podría realizarse con NLTK o spaCy también)
for _, row in df[:None].iterrows():
    sentence_tokens.append(text_to_word_sequence(row[0]))

I0000 00:00:1773982244.779035  413756 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1773982244.779414  413756 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1773982244.820202  413756 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1773982245.886473  413756 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

In [8]:
# Demos un vistazo
sentence_tokens[:2]

[['yesterday', 'all', 'my', 'troubles', 'seemed', 'so', 'far', 'away'],
 ['now', 'it', 'looks', 'as', 'though', "they're", 'here', 'to', 'stay']]

### 2 - Crear los vectores (word2vec)

In [9]:
from gensim.models.callbacks import CallbackAny2Vec


# Durante el entrenamiento gensim por defecto no informa el "loss" en cada época
# Sobrecargamos el callback para poder tener esta información
class Callback(CallbackAny2Vec):
    """
    Callback to print loss after each epoch
    """

    def __init__(self):
        self.epoch = 0

    def on_epoch_end(self, model):
        loss = model.get_latest_training_loss()
        if self.epoch == 0:
            print("Loss after epoch {}: {}".format(self.epoch, loss))
        else:
            print(
                "Loss after epoch {}: {}".format(
                    self.epoch, loss - self.loss_previous_step
                )
            )
        self.epoch += 1
        self.loss_previous_step = loss

In [10]:
# Crearmos el modelo generador de vectores
# En este caso utilizaremos la estructura modelo Skipgram
w2v_model = Word2Vec(
    min_count=5,  # frecuencia mínima de palabra para incluirla en el vocabulario
    window=2,  # cant de palabras antes y desp de la predicha
    vector_size=300,  # dimensionalidad de los vectores
    negative=20,  # cantidad de negative samples... 0: no se usa
    workers=2,  # si tienen más cores pueden cambiar este valor
    sg=1,
)  # modelo 0:CBOW  1:skipgram

In [11]:
# Obtener el vocabulario con los tokens
w2v_model.build_vocab(sentence_tokens)

In [12]:
# Cantidad de filas/docs encontradas en el corpus
print("Cantidad de docs en el corpus:", w2v_model.corpus_count)

Cantidad de docs en el corpus: 1846


In [13]:
# Cantidad de words encontradas en el corpus. Tamaño del vocabulario
print("Cantidad de words distintas en el corpus:", len(w2v_model.wv.index_to_key))

Cantidad de words distintas en el corpus: 445


📝 Resumen hasta el momento: 
- se dividio el dataset (canciones) en frases donde cada frase es un documento.
- se dividio cada documento en palabras (tokenización por palabra)
- Mediante Word2Vec se genero el vocabulario (445 palabras) excluyendo aquellas que aparecían menos de 5 veces (min_count=5)

### 3 - Entrenar embeddings

In [14]:
# Entrenamos el modelo generador de vectores
# Utilizamos nuestro callback
w2v_model.train(
    sentence_tokens,  # dataset
    total_examples=w2v_model.corpus_count,  # conteo del corpus
    epochs=40,
    compute_loss=True,
    callbacks=[Callback()],
)

Loss after epoch 0: 75194.125
Loss after epoch 1: 45665.5703125
Loss after epoch 2: 46118.5234375
Loss after epoch 3: 45729.25
Loss after epoch 4: 44736.015625
Loss after epoch 5: 44320.171875
Loss after epoch 6: 44395.5625
Loss after epoch 7: 44022.65625
Loss after epoch 8: 41544.1875
Loss after epoch 9: 39863.9375
Loss after epoch 10: 39438.96875
Loss after epoch 11: 38760.09375
Loss after epoch 12: 37399.75
Loss after epoch 13: 35990.625
Loss after epoch 14: 35459.375
Loss after epoch 15: 34896.6875
Loss after epoch 16: 34367.125
Loss after epoch 17: 33662.6875
Loss after epoch 18: 32916.0625
Loss after epoch 19: 32069.625
Loss after epoch 20: 32033.25
Loss after epoch 21: 31168.0
Loss after epoch 22: 30968.125
Loss after epoch 23: 30945.75
Loss after epoch 24: 30516.1875
Loss after epoch 25: 29609.1875
Loss after epoch 26: 30107.625
Loss after epoch 27: 27568.25
Loss after epoch 28: 27572.0
Loss after epoch 29: 26985.625
Loss after epoch 30: 26824.5
Loss after epoch 31: 27016.125
L

(313885, 575480)

📝 Para pocas epocas, el modelo puede no aprender lo suficiente. Pero para muchas epocas, el modelo puede sobreajustar.

### 4 - Ensayar

In [15]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["darling"], topn=10)

[('cry', 0.7375174164772034),
 ('seems', 0.7147207260131836),
 ('pretty', 0.7127755880355835),
 ('sleep', 0.6976456642150879),
 ('since', 0.6657601594924927),
 ('harm', 0.6400039792060852),
 ('smiles', 0.6307386159896851),
 ('show', 0.6222519278526306),
 ('does', 0.612738311290741),
 ('right', 0.6055706739425659)]

In [16]:
# Palabras que MENOS se relacionan con...:
w2v_model.wv.most_similar(negative=["love"], topn=10)

[('these', -0.12860804796218872),
 ('blackbird', -0.13291414082050323),
 ('said', -0.13686735928058624),
 ('behind', -0.13827665150165558),
 ('here', -0.141575887799263),
 ('words', -0.14649787545204163),
 ('wisdom', -0.15117935836315155),
 ('hammer', -0.160879984498024),
 ('it', -0.1645396649837494),
 ('singing', -0.16773533821105957)]

In [17]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["four"], topn=10)

[('five', 0.9050748944282532),
 ('sixty', 0.8757914900779724),
 ('six', 0.8738747835159302),
 ('seven', 0.8655507564544678),
 ('three', 0.8355680108070374),
 ('two', 0.8112839460372925),
 ('until', 0.7811483144760132),
 ('tight', 0.7313345670700073),
 ('feeling', 0.7265563011169434),
 ('understand', 0.682192325592041)]

In [18]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["money"], topn=5)

[('much', 0.8266410231590271),
 ('buy', 0.8233521580696106),
 ('thing', 0.7938603162765503),
 ('than', 0.7754770517349243),
 ("can't", 0.7152448892593384)]

In [19]:
# Ensayar con una palabra que no está en el vocabulario:
w2v_model.wv.most_similar(negative=["diedaa"])

KeyError: "Key 'diedaa' not present in vocabulary"

In [20]:
# el método `get_vector` permite obtener los vectores:
vector_love = w2v_model.wv.get_vector("love")
print(vector_love)

[-0.0755486   0.03743574 -0.06465601 -0.051126   -0.1731501  -0.19767116
 -0.2350349   0.22919935 -0.07265406  0.00845073  0.04750579 -0.14024505
 -0.03553869  0.17032917 -0.21432301 -0.21390377  0.10194691  0.04664158
 -0.11706302 -0.14599517 -0.13596706  0.12181655 -0.08661123 -0.17162369
 -0.06964035 -0.14708613  0.09192551  0.04395376  0.06117269 -0.2521305
 -0.10964326  0.06055963  0.13765378  0.32415998 -0.31737798  0.17564239
  0.32900822 -0.06022463 -0.12302988 -0.1487986   0.16924961 -0.12301028
 -0.06040367  0.09982729  0.20264915  0.1144781  -0.22427282 -0.00797571
  0.11189849 -0.1921853  -0.29563233 -0.07752367  0.14694786  0.24027522
  0.05724683  0.05374267  0.28514647  0.22478694  0.30862764  0.03558369
  0.18189384 -0.31962606 -0.16877751 -0.11043108 -0.01501738  0.08412141
  0.24874653  0.39552218 -0.19187078 -0.03848739  0.03380355 -0.10151386
  0.23760767 -0.24159224  0.01719509  0.34950617  0.01365145  0.00201908
 -0.13531971  0.20319478 -0.02323821 -0.07588745  0.

In [21]:
# el método `most_similar` también permite comparar a partir de vectores
w2v_model.wv.most_similar(vector_love)

[('love', 1.0),
 ('babe', 0.8008319139480591),
 ('hope', 0.7572897672653198),
 ('someone', 0.7163133025169373),
 ('need', 0.7088800668716431),
 ('whoa', 0.7027304768562317),
 ('somebody', 0.6761857867240906),
 ("somethin'", 0.6726495623588562),
 ('buy', 0.6719675064086914),
 ('michelle', 0.6651012897491455)]

In [22]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["love"], topn=10)

[('babe', 0.8008319735527039),
 ('hope', 0.7572897672653198),
 ('someone', 0.7163133025169373),
 ('need', 0.7088801264762878),
 ('whoa', 0.7027304768562317),
 ('somebody', 0.6761857867240906),
 ("somethin'", 0.6726495623588562),
 ('buy', 0.6719675064086914),
 ('michelle', 0.6651012897491455),
 ('show', 0.6645395755767822)]

📝 Como el dataset es chico, no se espera que el modelo aprenda mucho.

### 5 - Visualizar agrupación de vectores

In [23]:
from sklearn.manifold import TSNE
import numpy as np


def reduce_dimensions(model, num_dimensions=2):
    vectors = np.asarray(model.wv.vectors)
    labels = np.asarray(model.wv.index_to_key)

    tsne = TSNE(n_components=num_dimensions, random_state=0)
    vectors = tsne.fit_transform(vectors)

    return vectors, labels

In [24]:
# Graficar los embedddings en 2D
import plotly.express as px

vecs, labels = reduce_dimensions(w2v_model)

MAX_WORDS = 200
fig = px.scatter(x=vecs[:MAX_WORDS, 0], y=vecs[:MAX_WORDS, 1], text=labels[:MAX_WORDS])
# fig.show(renderer="colab") # esto para plotly en colab
fig.show()

In [ ]:
# Graficar los embedddings en 3D

vecs, labels = reduce_dimensions(w2v_model, 3)

fig = px.scatter_3d(
    x=vecs[:MAX_WORDS, 0],
    y=vecs[:MAX_WORDS, 1],
    z=vecs[:MAX_WORDS, 2],
    text=labels[:MAX_WORDS],
)
fig.update_traces(marker_size=2)
# fig.show(renderer="colab") # esto para plotly en colab
fig.show()

In [27]:
# También se pueden guardar los vectores y labels como tsv para graficar en
# http://projector.tensorflow.org/


vectors = np.asarray(w2v_model.wv.vectors)
labels = list(w2v_model.wv.index_to_key)

np.savetxt("vectors.tsv", vectors, delimiter="\t")

with open("labels.tsv", "w") as fp:
    for item in labels:
        fp.write("%s\n" % item)

### Consigna del desafío 2

**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado**

Recuerden que su notebook de entrega debe poder correrse de inicio a fin sin la aparición de errores.

- Crear sus propios vectores con Gensim basado en lo visto en clase con otro artista del dataset Songs.
- Elegir términos de interés y buscar términos más similares y menos similares.
- Realizar una reduccion de dimensionalidad a los embeddings, llevándolos a 2 dimensiones. Graficar los embeddings proyectados y seleccionar una cantidad de términos (variable MAX_WORDS) de forma tal que la visualización sea adecuada.
- Inspeccionar el grafico y buscar pequeños grupos de palabras que puedan formarse. Interpretarlos e intentar obtener conclusiones. En lo posible, acompañar los grupos de palabras con capturas (y pegarlas en celdas de texto)